# Mestrado — Proposta 1 nas cinco áreas (A0–A4)

Este notebook reúne **o treinamento original na Area1 (A0)** com a aplicação dos **mesmos oito classificadores fixos** a Sagrada Família (A1), Centro (A2), Lomba Grande (A3) e Cristo Rei (A4).

**Dados de referência:** `Arvores.gpkg` (257 pontos) e `Grama.gpkg` (253 pontos) pertencem à **Area1**. Os 510 pontos são usados **somente para o treinamento e validação espacial interna**. Não são pontos independentes dos municípios externos. Não inventar F1-macro.

**Resultados existentes:** as quatro imagens regionais já foram segmentadas integralmente por Quickshift em GitHub Actions, com resultados salvos no Drive. Este notebook executa a etapa supervisionada e prepara rótulos externos independentes.

**Limite científico:** ter uma ortoimagem de quatro bandas **não comprova NIR**. O processo interrompe a aplicação regional quando a quarta banda não estiver claramente identificada como NIR nos metadados **ou** por documento técnico cuja referência seja registrada. Nunca habilite a confirmação sem documentação.

**Execução:** abra `Ambiente de execução → Executar tudo`; a montagem do Drive depende de sua autorização na sessão Colab. A execução de imagens de vários gigabytes pode ultrapassar a cota de uma sessão. Os checkpoints e modelos ficam no Drive para retomada posterior.

## 1. Autorizar o Google Drive e obter o código atualizado

In [ ]:
import sys, subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
REPO=Path('/content/mestrado-arvores-urbanas-externo')
BRANCH='feature/validacao-geografica-externa'
if not REPO.is_dir():
    subprocess.run(['git','clone','--depth=1','--branch',BRANCH,
        'https://github.com/BrenoBragaGalvao/mestrado-arvores-urbanas.git',str(REPO)],
        check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',BRANCH],check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin',BRANCH],check=True)
SCRIPTS=REPO/'experimentos/validacao_externa_regioes'
# Cristo Rei usa RAR: o Colab precisa do executável 7z, não só de rarfile.
import shutil
if shutil.which('7z') is None:
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','-qq','p7zip-full'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',
    str(SCRIPTS/'requirements.txt')],check=True)
print('Código utilizado:',SCRIPTS)
print('Versão Git:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

## 2. Verificação da Area1 e parâmetros de execução

Primeiro será executada a auditoria, reconstrução dos **505 segmentos rotulados**, ajuste dos **oito modelos** e validação aninhada com cinco divisões por escala. Depois, uma aplicação dos modelos à Area1 gerará mapas **descritivos, não um novo teste independente**.

Os quatro pacotes regionais são processados **um por vez**, apagando apenas os arquivos temporários de cada região no Colab após salvar os resultados na pasta de saída do Drive.

Os arquivos originais e o README principal **não serão editados**.

In [ ]:
ROOT=Path('/content/drive/MyDrive')
AREA1=ROOT/'Projetos/Segmentacao_Breno/Area1'
OUTPUT=ROOT/'Projetos/Segmentacao_Breno/outputs/validacao_externa_regioes'
PACKAGES=ROOT/'Imagens do Mestrado'
LOCAL=Path('/content/validacao_externa_local')
FONTE_NIR=OUTPUT/'confirmacoes_nir.json'
REGIOES=[
 'sagrada_familia_caxias',
 'centro_novo_hamburgo',
 'lomba_grande_novo_hamburgo',
 'cristo_rei_sao_leopoldo',
]
SOMENTE_AREA1=False  # True limita a execução aos modelos e mapas da Area1
GERAR_MAPAS_AREA1=True
QUADRANTES_TESTE=None  # None = processamento integral; 2 = apenas teste técnico
OUTPUT.mkdir(parents=True,exist_ok=True)
assert (AREA1/'AOI_RGB_NEW.tif').is_file(), f'Não encontrei RGB da Area1: {AREA1}'
assert (AREA1/'AOI_NEW_MULTI.tif').is_file(), 'Falta raster multiespectral da Area1.'
assert (AREA1/'Arvores.gpkg').is_file() and (AREA1/'Grama.gpkg').is_file(), 'Faltam os 510 pontos originais.'
print('Area1 localizada em:',AREA1)
print('Documentação NIR:', FONTE_NIR if FONTE_NIR.is_file() else 'pendente; aplicação externa será bloqueada sem prova')
import shutil
print('Espaço livre no Colab (GB):',round(shutil.disk_usage('/content').free/1e9,2))

## 3. Como registrar a prova de NIR (apenas se houver documentação)

Os metadados dos TIFFs podem identificar explicitamente a quarta banda. Nesse caso, a verificação será automática.

Se **não** identificarem, obtenha um documento do fornecedor do levantamento comprovando que a quarta banda é infravermelho próximo e registre sua fonte em `confirmacoes_nir.json`, na pasta de resultados do Drive, no formato abaixo. **Não marque `confirmada: true` sem evidência.**

```json
{
  "sagrada_familia_caxias": {
    "confirmada": false,
    "fonte": ""
  },
  "centro_novo_hamburgo": {
    "confirmada": false,
    "fonte": ""
  },
  "lomba_grande_novo_hamburgo": {
    "confirmada": false,
    "fonte": ""
  },
  "cristo_rei_sao_leopoldo": {
    "confirmada": false,
    "fonte": ""
  }
}
```

O código **não altera** automaticamente esse documento nem considera o nome `4BANDAS` prova de NIR.

## 4. Executar o fluxo completo — A0 (Area1) + A1, A2, A3 e A4

In [ ]:
args=[sys.executable,str(SCRIPTS/'executar_proposta1_cinco_areas.py'),
      '--drive-root',str(ROOT),
      '--area1',str(AREA1),
      '--pacotes-drive',str(PACKAGES),
      '--out',str(OUTPUT),
      '--scratch',str(LOCAL),
      '--regioes',*REGIOES]
if SOMENTE_AREA1:
    args+=['--somente-area1']
if not GERAR_MAPAS_AREA1:
    args+=['--pular-area1-mapas']
if QUADRANTES_TESTE is not None:
    args+=['--limite-quadrantes',str(QUADRANTES_TESTE)]
if FONTE_NIR.is_file():
    args+=['--confirmacoes-nir',str(FONTE_NIR)]
print('Iniciando a execução. As saídas serão guardadas em:',OUTPUT)
subprocess.run(args,check=True)
print('Processamento da etapa solicitado concluído sem erros do orquestrador.')

## 5. Conferir o que realmente foi executado

O relatório de estado separa **treinamento**, **predição** e **validação externa com rótulos independentes**. A ausência de F1-macro externo é esperada onde ainda não existirem referências locais.

A execução não deve ser descrita como concluída nas cinco áreas se algum processamento estiver `parcial`, `bloqueado_confirmacao_nir` ou `erro`.

In [ ]:
import json
import pandas as pd
from IPython.display import display
STATE=OUTPUT/'estado_cinco_areas.json'
if STATE.is_file():
    states=json.loads(STATE.read_text(encoding='utf-8'))
    display(pd.DataFrame([
        {'area':key,'estado':v.get('estado'),
         'observacao':str(v.get('motivo',v.get('nota','')))[:160]}
        for key,v in states.items() if isinstance(v,dict)
    ]))
    for area in ['area1_caxias',*REGIOES]:
        summary=OUTPUT/'predicoes'/area/'resumo_classes_previstas.csv'
        if summary.is_file():
            print('\nPredições por modelo:',area)
            display(pd.read_csv(summary))
        score=OUTPUT/'metricas'/area/'metricas_externas.csv'
        if score.is_file():
            print('\nF1 externo com rótulos independentes:',area)
            display(pd.read_csv(score))
else:
    print('Ainda não foi gerado um relatório de execução.')
REPORT=OUTPUT/'comparacao_regioes'/'relatorio.md'
print('Relatório comparativo:',REPORT if REPORT.is_file() else 'ainda não produzido')

## 6. Rotulação externa e continuidade

Quando os mapas estiverem completos e não existirem referências locais, será criado `rotulacao/REGIAO/amostras_para_anotar.gpkg`.

**Amostras para Sagrada Família:** a Area1 e uma faixa de 25 m são excluídas da amostragem. Os 510 pontos originais não são reutilizados como teste.

Abra cada arquivo de amostras no QGIS **com a ortoimagem RGB original, sem exibir as previsões dos classificadores**, e preencha `classe_verificada` com `arvore`, `grama` ou `incerto`. Depois exporte os rótulos com:

```bash
python preparar_rotulacao.py exportar --anotadas amostras_para_anotar.gpkg --out rotulos_verificados
```

Rode este notebook novamente. O orquestrador reconhecerá os mapas completos, verificará os rótulos disponíveis e calculará o **F1-macro externo** somente quando houver amostras independentes adequadas. As comparações históricas da Area1 serão descritas como contextuais, não como testes pareados.

**Não há execução remota da sessão Colab a partir desta conversa.** O notebook é o ponto de entrada para você executar na sessão que já autorizou.